In [ ]:
from google.colab import drive
drive.mount("/content/gdrive", force_remount=True)

In [ ]:
import sys
import os
working_path = '???'
sys.path.append(working_path)
os.chdir(working_path)

In [ ]:
pip install icrawler

In [ ]:
query_list = ['apple', 'banana', 'coconut fruits']
DATA_NAME = 'fruits/'

MAX_NUM_IMAGES = 200
class_folder_list = []
for var in query_list:
  var_folder = var.replace(" ", "_")
  class_folder_list.append(var_folder)

DATASET_PATH = 'data/' + DATA_NAME + 'dataset/'
image_path = working_path + DATASET_PATH

In [ ]:
# Images originally saved in folders with the following patterns
# data_path/
#     class0/
#         classs0_0.jpg
#         classs0_1.jpg
#         ...
#         classs0_k.jpg
#     class1/
#         classs1_0.jpg
#         classs1_1.jpg
#         ...
#         classs1_k.jpg
# ...
#     classN/
#         classsN_0.jpg
#         classsN_1.jpg
#         ...
#         classsN_k.jpg
# npy/
#     X_train.npy
#     y_train.npy
#     X_test.npy
#     y_test.npy

In [ ]:
# Download images according to var
# Images saved under working_path + data_path + var
selected_class_id = 0
var = query_list[selected_class_id]
var_folder = class_folder_list[selected_class_id]

path = image_path + var_folder
print(path)

from icrawler.builtin import BingImageCrawler
bing_crawler = BingImageCrawler(storage={'root_dir': path})
bing_crawler.crawl(keyword=var, max_num=MAX_NUM_IMAGES)

In [ ]:
# Download images according to var
# Images saved under working_path + data_path + var
selected_class_id = 1
var = query_list[selected_class_id]
var_folder = class_folder_list[selected_class_id]

path = image_path + var_folder
print(path)

from icrawler.builtin import BingImageCrawler
bing_crawler = BingImageCrawler(storage={'root_dir': path})
bing_crawler.crawl(keyword=var, max_num=MAX_NUM_IMAGES)

In [ ]:
# Download images according to var
# Images saved under working_path + data_path + var
selected_class_id = 2
var = query_list[selected_class_id]
var_folder = class_folder_list[selected_class_id]

path = image_path + var_folder
print(path)

from icrawler.builtin import BingImageCrawler
bing_crawler = BingImageCrawler(storage={'root_dir': path})
bing_crawler.crawl(keyword=var, max_num=MAX_NUM_IMAGES)

In [ ]:
# Some images are corrupted.
# Remove them from the image directory.
import os
import tensorflow as tf

num_skipped = 0
for folder_name in class_folder_list:
    folder_path = os.path.join(image_path, folder_name)
    print(folder_path)
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        try:
            fobj = open(file_path, "rb")
            is_jfif = tf.compat.as_bytes("JFIF") in fobj.peek(10)
        finally:
            fobj.close()

        if not is_jfif:
            num_skipped += 1
            # Delete corrupted image
            os.remove(file_path)

print("Deleted %d images" % num_skipped)

In [ ]:
# Rename files, so they start with class name
for folder_name in class_folder_list:
    folder_path = os.path.join(image_path, folder_name)
    file_count = 0
    for filename in os.listdir(folder_path):
        old_path = os.path.join(folder_path, filename)

        new_name = folder_name + '_' + str(file_count) + '.jpg'
        new_path = os.path.join(folder_path, new_name)

        os.rename(old_path, new_path)
        file_count += 1

### Rearrange Data into Train, Validation and Test subfolders

In [ ]:
# Images finally saved in folders with the following patterns
# data_path/
#     train/
#        class0/
#           classs0_0.jpg
#           ...
#           classs0_k.jpg
#       class1/
#         classs1_0.jpg
#         ...
#         classs1_k.jpg
#       ...
#       classN/
#         classsN_0.jpg
#         ...
#         classsN_k.jpg
#     validation/
#        class0/
#        ...
#        classN/
#     test/
#        class0/
#        ...
#        classN

In [ ]:
train_ratio = 0.8
val_ratio = 0.10
test_ratio = 0.10

In [ ]:
class_names = class_folder_list
class_names

In [ ]:
for split in ["train","validation","test"]:
    for cls in class_names:
        os.makedirs(os.path.join(DATASET_PATH, split, cls), exist_ok=True)

In [ ]:
import random
import shutil

for cls in class_names:

    src_folder = os.path.join(DATASET_PATH, cls)
    images = os.listdir(src_folder)

    random.shuffle(images)

    total = len(images)
    train_end = int(total * train_ratio)
    val_end = int(total * (train_ratio + val_ratio))

    train_imgs = images[:train_end]
    val_imgs = images[train_end:val_end]
    test_imgs = images[val_end:]

    for img in train_imgs:
        shutil.move(os.path.join(src_folder,img),
                    os.path.join(DATASET_PATH,"train",cls,img))

    for img in val_imgs:
        shutil.move(os.path.join(src_folder,img),
                    os.path.join(DATASET_PATH,"validation",cls,img))

    for img in test_imgs:
        shutil.move(os.path.join(src_folder,img),
                    os.path.join(DATASET_PATH,"test",cls,img))

In [ ]:
for cls in class_names:
    os.rmdir(os.path.join(DATASET_PATH, cls))